# 05 Stratified Analysis and Confounding

Ch03 found that shower users had an elevated RR, but a senior outbreak investigator suspected it was caused by **confounding**.

> 🌧️ **People who wear raincoats catch colds more often—so does wearing a raincoat give you a cold?** Of course not! It's because "rainy days" make you both put on a raincoat and catch a cold more easily. "Rainy day" is the confounder.

In our case, "functional status" is that "rainy day"—it affects both whether a resident uses the shower and their infection risk.

This lesson: **verify the confounder conditions → stratum-specific RR → forest plot → Mantel-Haenszel adjustment → test of homogeneity**.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: Data preparation ---
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import chi2_contingency
from epi_learning.metrics import risk_ratio

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
print(f"Total: {len(df)} people, infected: {df['infected'].sum()} people")

In [ ]:
# --- Step 2: Recap of the crude RR ---
ct = pd.crosstab(df["shower_use"], df["infected"])
a = int(ct.loc[1, 1])
b = int(ct.loc[1, 0])
c = int(ct.loc[0, 1])
d = int(ct.loc[0, 0])

crude_rr = risk_ratio(a, a + b, c, c + d)
print(f"Crude RR (shower_use -> infected) = {crude_rr:.3f}")
print(f"  Exposed group risk: {a/(a+b):.1%}  Unexposed group risk: {c/(c+d):.1%}")

## DAG (Directed Acyclic Graph)—a Map of Causation

We suspect `functional_status` is a confounder. A DAG uses arrows to show "who affects whom":

```
functional_status → shower_use → infected
functional_status ─────────────→ infected
```

- **Direct path** (the one we want to study): shower use → infection
- **Back-door path** (the confounding path): shower ← functional status → infection

> The back-door path is like a classmate next to you copying your answers during an exam—their score looks related to yours, but really it's because of the common cause of "sitting next to you." Stratified analysis separates "those sitting next to you" from "those not sitting next to you."

The three requirements of a confounder (missing even one disqualifies it):
1. C is associated with the **exposure** (only people who can walk use the shower)
2. C is associated with the **outcome** (people who can walk have a wider range of movement and more exposure)
3. C is **not** an intermediate step on the exposure→outcome path

### How Do You Discover Potential Confounders?

Our story opened with a "senior outbreak investigator" drawing on experience to suggest that functional status might be a confounder. But you can't rely on senior staff every time—is there a more systematic method?

| Method | How it works | Advantage | Limitation |
|------|------|------|------|
| **Literature review** | Search papers from similar past outbreak investigations to see which confounders others controlled for | Stand on the shoulders of predecessors | A novel disease may have no precedent |
| **Draw a DAG** | Draw a causal diagram based on domain knowledge and find the "back-door paths" | Logically clear; can distinguish confounders vs. intermediate variables | Requires a basic understanding of the causal mechanism |
| **Statistical screening** | Check whether a candidate variable is significantly associated with both the exposure and the outcome (requirements #1 + #2) | Backed by data, not purely intuition | Statistical significance ≠ causation |
| **Change-in-estimate** | Add/remove the candidate variable in a model and see whether the RR or OR changes by ≥ 10% | Directly answers "is it confounding or not" | Requires a regression model first (→ Ch06) |
| **Expert consultation** | Consult clinicians, infection-control staff, and senior epidemiologists | Captures practical factors that statistics miss | Subjective; may have omissions |

> 💡 **In practice, we recommend a three-pronged approach: "literature + DAG + statistical screening."** First review the literature to draw up a candidate list → draw a DAG to mark causal directions → use data to verify the three confounder requirements. Finally, have senior staff review it.

In [ ]:
# --- Step 3: Verify the three confounder requirements ---
# Before stratifying, confirm that "functional status" really meets the three confounder conditions

# Requirement 1: Is functional_status associated with shower_use?
print("=== Requirement 1: Functional status x Shower use rate ===")
print(pd.crosstab(df["functional_status"], df["shower_use"],
                  margins=True, normalize="index").round(3))

print()

# Requirement 2: Is functional_status associated with infected?
print("=== Requirement 2: Functional status x Infection rate ===")
print(pd.crosstab(df["functional_status"], df["infected"],
                  margins=True, normalize="index").round(3))

# Requirement 3 (judged by logic): functional status is not an intermediate step on the shower->infection path
# A person does not become ambulatory because they "used the shower first" -- the causal direction is wrong
# -> All three conditions are met, so we can proceed to stratified analysis!
print("\nRequirement 3: functional status is not on the shower->infection causal path ✓")
print("-> All three requirements are met; confirmed as a confounder")

In [ ]:
# --- Step 4: Stratified analysis ---
# Stratify by functional_status and compute the RR of shower_use within each stratum

strata = sorted(df["functional_status"].unique())
stratum_results = []

for s in strata:
    sub = df[df["functional_status"] == s]
    ct_s = pd.crosstab(sub["shower_use"], sub["infected"])

    if ct_s.shape != (2, 2):
        print(f"  {s}: skipped (missing some combinations)")
        continue

    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s

    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)

    # 95% CI
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)

    stratum_results.append({
        "stratum": s, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

results_df = pd.DataFrame(stratum_results)
print("=== Stratum-specific RR ===")
for _, row in results_df.iterrows():
    print(f"  {row['stratum']:20s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})  n={row['n']}")
print(f"\n  Crude RR = {crude_rr:.3f}")

In [ ]:
# --- Step 5: Forest plot ---
fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(results_df))

ax.errorbar(
    results_df["RR"], y_pos,
    xerr=[results_df["RR"] - results_df["CI_lower"],
          results_df["CI_upper"] - results_df["RR"]],
    fmt="o", color="#2c7fb8", capsize=4, markersize=8,
)
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=crude_rr, color="red", linestyle=":", alpha=0.7,
           label=f"Crude RR={crude_rr:.2f}")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(results_df["stratum"])
ax.set_xlabel("Risk Ratio (RR)")
ax.set_title("Stratified analysis forest plot: shower use -> infection (stratified by functional status)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 6: Mantel-Haenszel weighted RR ---
# Principle: strata with more people get more weight, strata with fewer get less
# It's like a semester grade -- you can't just average quizzes and the final exam; the final should count for more
numerator = 0
denominator = 0

for _, row in results_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh = numerator / denominator

print(f"Mantel-Haenszel adjusted RR = {rr_mh:.3f}")
print(f"Crude RR                    = {crude_rr:.3f}")
print(f"Difference                  = {crude_rr - rr_mh:.3f}")

# --- Use the 10% rule to judge whether confounding is present ---
# |crude RR - adjusted RR| / adjusted RR >= 10% -> confounding present
change_pct = abs(crude_rr - rr_mh) / rr_mh * 100
print(f"\nMagnitude of change = {change_pct:.1f}% (10% rule threshold)")
if change_pct >= 10:
    print("-> Change >= 10%, the crude RR was affected by confounding!")
    if crude_rr > rr_mh:
        print("  Direction of confounding: inflation (crude RR too high, like a burger stuffed with too much lettuce)")
    else:
        print("  Direction of confounding: suppression (crude RR too low, like an ice cube pressing on the thermometer)")
else:
    print("-> Change < 10%, confounding is not notable")

In [ ]:
# --- Step 7: Test of homogeneity -- is there interaction? ---
# Interaction (effect modification): the effect of exposure "varies from person to person"
# Analogy: investigating "whether eating spicy hotpot causes diarrhea"
#   strong-stomached people RR=1.2, weak-stomached people RR=4.5 -> this isn't confounding, it's interaction
#   you can't just report a pooled RR, you must report them separately

rr_values = results_df["RR"].values
rr_range = rr_values.max() - rr_values.min()

print("=== Homogeneity assessment ===")
for _, row in results_df.iterrows():
    print(f"  {row['stratum']:20s}  RR = {row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})")
print(f"\n  RR range: {rr_values.min():.3f} – {rr_values.max():.3f}")
print(f"  Spread: {rr_range:.3f}")

if rr_range > 0.5:
    print("\n-> The stratum-specific RRs differ substantially; effect modification may be present")
    print("  Like three branch stores with wildly different spice levels -- you can't give one average spiciness -> report each stratum's RR")
else:
    print("\n-> The stratum-specific RRs are similar; the MH weighted pooled value is reasonable to use")
    print("  Like three branch stores with consistent coffee flavor -> a single brand average rating is enough")

In [ ]:
# --- Step 8: A second example - stratifying by floor ---
print("=== Stratified analysis by floor: shower -> infection ===")

for floor in sorted(df["floor"].unique()):
    sub = df[df["floor"] == floor]
    ct_f = pd.crosstab(sub["shower_use"], sub["infected"])
    if ct_f.shape != (2, 2):
        print(f"  {floor}F: skipped")
        continue
    a_f, b_f = int(ct_f.loc[1, 1]), int(ct_f.loc[1, 0])
    c_f, d_f = int(ct_f.loc[0, 1]), int(ct_f.loc[0, 0])
    rr_f = risk_ratio(a_f, a_f + b_f, c_f, c_f + d_f)
    print(f"  {floor}F: RR={rr_f:.3f}  "
          f"(shower: {a_f}/{a_f+b_f}, no shower: {c_f}/{c_f+d_f})")

print(f"\n  Crude RR = {crude_rr:.3f}")

## Supplement: Can Case-Control Studies Also Use Stratified Analysis?

Our nursing home data is a **cohort study**—we followed all 280 residents and used attack rates to compute RR. But in a **case-control study** (pick cases + controls, then ask retrospectively about exposure history), you can't compute an attack rate, so you **can only compute an OR (odds ratio)**.

Good news: **the logic of stratified analysis is exactly the same**—verify the three requirements, stratify by the confounder, pool with MH. The only difference:

| | Cohort study (this chapter) | Case-control study |
|---|---|---|
| **Effect measure** | RR (risk ratio) | OR (odds ratio) |
| **Per-stratum calculation** | RR = [a/(a+b)] / [c/(c+d)] | OR = (a·d) / (b·c) |
| **MH pooling** | RR_MH = Σ[a·(c+d)/N] / Σ[c·(a+b)/N] | OR_MH = Σ(a·d/N) / Σ(b·c/N) |
| **Judging confounding** | crude RR vs. adjusted RR (10% rule) | crude OR vs. adjusted OR (10% rule) |

> 💡 **Mnemonic**: cohort → MH adjusted **RR**; case-control → MH adjusted **OR**. Same method, just a different effect measure. When the attack rate is low (< 10%), OR ≈ RR; when it's high (like 43% in this case), the OR overestimates—Ch06 will explore this in depth.

## Summary

| Step | Skill learned | In plain language |
|------|------------|--------|
| Three confounder requirements | Verify the C-exposure and C-outcome associations | Confirm the "double agent's" identity |
| Stratum-specific RR | `crosstab` + a loop to compute each stratum's RR | Compare high heat and low heat separately, locking down the heat |
| Forest plot | `errorbar` to visualize each stratum's effect | See each stratum's RR and precision at a glance |
| MH adjustment | Compute the Mantel-Haenszel RR by hand | A "fair pooling" weighted by number of people |
| 10% rule | \|crude RR − adjusted RR\| / adjusted RR | Judge whether confounding is big enough to affect the conclusion |
| Homogeneity | Compare stratum RRs → judge interaction | Do the three branch stores taste the same? |

**Limitation**: stratified analysis can control for only one confounder at a time. What if you have several confounders at once—age, functional status, comorbidities?
→ Ch06's **Modified Poisson regression** can adjust for all variables at once and directly compute an **adjusted RR**. At the same time, it uses logistic regression as a comparison so you can see how much the OR overestimates at a high attack rate.